# 01b: 特徴量エンジン検証

実データで特徴量エンジン(`src/features/`)を動作させ、各特徴量の品質を確認する。

In [9]:
import os
os.environ["PGPASSWORD"] = "aa8940aa"

import sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np
import pandas as pd

fm.fontManager.addfont(r"C:\Windows\Fonts\YuGothM.ttc")
plt.rcParams["font.family"] = "Yu Gothic"
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["figure.dpi"] = 120

from db.connection import DatabaseConnection
from features.feature_engine import FeatureEngine
from features.leakage_validators import validate_no_future_leakage

conn = DatabaseConnection()
engine = conn.get_engine()
print("Setup complete")

Setup complete


In [10]:
print("データをロード中...")
race_df = conn.load_races("20230101", "20261231")
entry_df = conn.load_entries_with_results("20230101", "20261231")
odds_df = conn.load_odds_snapshots("20230101", "20261231")

# 時系列オッズは80M行あるため直近1年分のみ
odds_ts_df = conn.load_odds_time_series_range("20240101", "20261231")

print(f"  races: {len(race_df):,}")
print(f"  entries: {len(entry_df):,}")
print(f"  odds_snapshots: {len(odds_df):,}")
print(f"  odds_timeseries: {len(odds_ts_df):,}")

データをロード中...
  races: 10,826
  entries: 148,327
  odds_snapshots: 149,184
  odds_timeseries: 16,336,438


In [11]:
print("特徴量を生成中...")
feat_engine = FeatureEngine()
feat_df = feat_engine.build_all(race_df, entry_df, odds_df, odds_ts_df=odds_ts_df)
print(f"  生成データ: {len(feat_df):,} 行 × {len(feat_df.columns)} 列")

特徴量を生成中...
  生成データ: 148,327 行 × 53 列


In [12]:
print("=== 欠損値チェック ===")
nulls = feat_df.isnull().sum()
null_cols = nulls[nulls > 0].sort_values(ascending=False)
if len(null_cols) == 0:
    print("  欠損値なし ✓")
else:
    for col, cnt in null_cols.items():
        pct = cnt / len(feat_df) * 100
        print(f"  {col:40s} {cnt:>8,} ({pct:.1f}%)")

=== 欠損値チェック ===
  odds_drop_rate_60_10                      126,503 (85.3%)
  odds_velocity                              45,943 (31.0%)
  popularity_change_30_10                    45,903 (30.9%)
  odds_drop_rate_30_10                       45,903 (30.9%)
  odds_volatility                            45,903 (30.9%)
  zogen_sa                                   15,394 (10.4%)
  tan_odds                                      322 (0.2%)
  place_odds_actual                             322 (0.2%)
  fuku_odds                                     322 (0.2%)
  p_market_win_adj                              322 (0.2%)


In [13]:
import numpy as np
print("=== Inf値チェック ===")
numeric_cols = feat_df.select_dtypes(include=[np.number]).columns
inf_cols = []
for col in numeric_cols:
    if np.isinf(feat_df[col]).any():
        inf_cols.append((col, np.isinf(feat_df[col]).sum()))
if not inf_cols:
    print("  Inf値なし ✓")
else:
    for col, cnt in inf_cols:
        print(f"  {col}: {cnt} 件")

=== Inf値チェック ===
  Inf値なし ✓


In [14]:
print("=== 特徴量の基本統計 ===")
feature_cols = [c for c in feat_df.columns if c not in [
    "race_id", "race_date", "umaban", "ketto_num", "year", "month_day",
    "jyo_cd", "kaiji", "nichiji", "race_num", "surface", "surface_key"
]]
feat_df[feature_cols].describe().T

=== 特徴量の基本統計 ===


,count,mean,min,25%,50%,75%,max,std
year_x,148327.0,2024.147721,2023.0,2023.0,2024.0,2025.0,2026.0,0.943842
track_cd,148327.0,19.539646,10.0,17.0,23.0,24.0,24.0,4.739052
distance,148327.0,1620.703244,1000.0,1400.0,1600.0,1800.0,3600.0,347.595483
tenko_cd,148327.0,1.532007,1.0,1.0,1.0,2.0,6.0,0.803411
track_condition_code,148327.0,1.413633,1.0,1.0,1.0,2.0,4.0,0.771875
field_size,148327.0,14.442003,5.0,13.0,15.0,16.0,18.0,2.566908
race_date_x,148327,2024-08-10 17:53:52.298906880,2023-01-05 00:00:00,2023-10-21 00:00:00,2024-08-10 00:00:00,2025-06-01 00:00:00,2026-03-22 00:00:00,NaN
finish_pos,148327.0,7.69698,1.0,4.0,7.0,11.0,18.0,4.404262
finish_time,148327.0,1464.571117,537.0,1211.0,1392.0,1562.0,3509.0,369.796126
haron_time_l3,148327.0,370.746209,313.0,353.0,369.0,385.0,999.0,23.175538


In [15]:
# リーク検証はhist系特徴量用（現在のデータにはhist_*が含まれないためスキップ）
print("=== リーク検証 ===")
hist_cols = [c for c in feat_df.columns if c.startswith("hist_")]
if hist_cols:
    print(f"  hist系特徴量: {hist_cols}")
else:
    print("  hist系特徴量なし — リーク検証対象なし ✓")

=== リーク検証 ===
  hist系特徴量なし — リーク検証対象なし ✓
